In [2]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_bureau")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b6e81674-8899-4adf-a885-090d26ef319a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 169ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [3]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [4]:
path = "s3a://bronze/base_score_bureau_movel/"
df_base_score_bureau_movel = spark.read.parquet(path)
df_base_score_bureau_movel.cache()
df_base_score_bureau_movel.show(20, truncate=False)

26/01/01 14:22:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------+---------------+---+----+---------+--------+--------+-----------+
|SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|NUM_CPF    |
+------+---------------+---+----+---------+--------+--------+-----------+
|202410|1              |0  |CMV |PRE      |562     |636     |ZZZZZX7XWY8|
|202410|1              |1  |CMV |PRE      |546     |518     |ZZZZZX88YXY|
|202410|1              |0  |CMV |PRE      |621     |750     |ZZZZZYT7XYT|
|202410|1              |1  |CMV |PRE      |609     |679     |ZZZZZNTXY9Z|
|202410|1              |0  |CMV |PRE      |621     |722     |ZZZZZ79ZXUX|
|202410|1              |0  |CMV |PRE      |614     |635     |ZZZZZ8YYWNX|
|202410|1              |0  |CMV |PRE      |578     |586     |ZZZZXZ8979Z|
|202410|1              |0  |CMV |PRE      |602     |701     |ZZZZXXUWT9Z|
|202410|1              |0  |CMV |PRE      |638     |772     |ZZZZXYYU8WY|
|202410|1              |0  |CMV |PRE      |542     |591     |ZZZZXN8TYXZ|
|202410|1              |0  |CMV |PRE  

In [5]:
df_base_score_bureau_movel.createOrReplaceTempView("raw_00")

In [7]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20, truncate=False)

+------+------------+-------------+
|SAFRA |total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|203828      |203828       |
|202411|227176      |227176       |
|202412|227985      |227985       |
|202501|221002      |221002       |
|202502|203139      |203139       |
|202503|207396      |207396       |
+------+------------+-------------+



In [8]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM raw_00
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM raw_00 r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [9]:
df_resultado = contagem_percentual("PROD")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|CMV         |1290526      |100.00       |
+------------+-------------+-------------+



In [10]:
df_resultado = contagem_percentual("flag_mig2")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|PRE         |1290526      |100.00       |
+------------+-------------+-------------+



In [11]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('try_cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
try_cast(SAFRA as) as SAFRA,
try_cast(FLAG_INSTALACAO as) as FLAG_INSTALACAO,
try_cast(FPD as) as FPD,
try_cast(PROD as) as PROD,
try_cast(flag_mig2 as) as flag_mig2,
try_cast(SCORE_01 as) as SCORE_01,
try_cast(SCORE_02 as) as SCORE_02,
try_cast(NUM_CPF as) as NUM_CPF,


In [12]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            cast(NUM_CPF as string) as NUM_CPF,
            cast(SAFRA as int) as SAFRA,
            cast(FLAG_INSTALACAO as int) as FLAG_INSTALACAO,
            cast(FPD as int) as FPD,
            cast(PROD as string) as PROD,
            cast(flag_mig2 as string) as flag_mig2,
            cast(SCORE_01 as int) as SCORE_01,
            cast(SCORE_02 as int) as SCORE_02,
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.cache()
lake.count()  

1290526

In [13]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- FLAG_INSTALACAO: integer (nullable = true)
 |-- FPD: integer (nullable = true)
 |-- PROD: string (nullable = true)
 |-- flag_mig2: string (nullable = true)
 |-- SCORE_01: integer (nullable = true)
 |-- SCORE_02: integer (nullable = true)
 |-- DATPROC: long (nullable = false)



In [14]:
lake.show(5, truncate=False)

+-----------+------+---------------+---+----+---------+--------+--------+--------------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|DATPROC       |
+-----------+------+---------------+---+----+---------+--------+--------+--------------+
|ZZZZZX7XWY8|202410|1              |0  |CMV |PRE      |562     |636     |20260101142217|
|ZZZZZX88YXY|202410|1              |1  |CMV |PRE      |546     |518     |20260101142217|
|ZZZZZYT7XYT|202410|1              |0  |CMV |PRE      |621     |750     |20260101142217|
|ZZZZZNTXY9Z|202410|1              |1  |CMV |PRE      |609     |679     |20260101142217|
|ZZZZZ79ZXUX|202410|1              |0  |CMV |PRE      |621     |722     |20260101142217|
+-----------+------+---------------+---+----+---------+--------+--------+--------------+
only showing top 5 rows



In [15]:
for col in lake.columns:
    agg_result = lake.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = lake.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

SCORE_01: 9439 nulos (0.73%)
SCORE_02: 576 nulos (0.04%)


In [16]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, SAFRA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup.cache()
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

1290526

In [17]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20)

+------+------------+-------------+
| SAFRA|total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|      203828|       203828|
|202411|      227176|       227176|
|202412|      227985|       227985|
|202501|      221002|       221002|
|202502|      203139|       203139|
|202503|      207396|       207396|
+------+------------+-------------+



In [18]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_score_bureau_movel/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )
    print("Dados inseridos com sucesso...")

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")

Tabela silver não existe. Criando...


In [19]:
name = "base_score_bureau_movel"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM lake_dedup
    GROUP BY SAFRA
""".format(name_table=name))

df_controle.show()

+--------------------+------+-------------+--------------------+
|         nome_tabela| safra|qtd_registros|             datproc|
+--------------------+------+-------------+--------------------+
|base_score_bureau...|202501|       221002|2026-01-01 14:26:...|
|base_score_bureau...|202412|       227985|2026-01-01 14:26:...|
|base_score_bureau...|202502|       203139|2026-01-01 14:26:...|
|base_score_bureau...|202503|       207396|2026-01-01 14:26:...|
|base_score_bureau...|202411|       227176|2026-01-01 14:26:...|
|base_score_bureau...|202410|       203828|2026-01-01 14:26:...|
+--------------------+------+-------------+--------------------+



In [20]:
silver_controle_path = "s3a://silver/controle/"
if not DeltaTable.isDeltaTable(spark, silver_controle_path):
    print("Tabela de controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela de controle existe. Inserindo novo registro...")

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")

Tabela de controle existe. Inserindo novo registro...


In [21]:
spark.stop()